In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
from rdkit.Chem import AllChem
import os
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import shutil

In [ ]:

df = pd.read_excel('HTQC_reaction_thermo.xlsx')
mol_excel_path = 'HTQC_reaction_thermo.xlsx'

In [ ]:
df

In [ ]:

nproc = 10
mem = 2000 

In [ ]:
def check_and_fill_eps(df):
    
    
    if 'eps' in df.columns:
        df['eps'] = df['eps'].fillna(0)  
    else:
        print("Public message removed for release.")
    
    
    df.to_excel('HTQC_reaction_thermo.xlsx', index=False)

    
    return df

In [ ]:
df = check_and_fill_eps(df)
df

In [ ]:

def extract_mol_system_dict(df):
    
    reactant_name_dict = {}
    product_name_dict = {}

    
    reactant_name_cols = [col for col in df.columns if col.startswith('Reactant_Name_')]
    reactant_smiles_cols = [col for col in df.columns if col.startswith('Reactant_SMILES_')]

    for name_col, smiles_col in zip(reactant_name_cols, reactant_smiles_cols):
        reactant_name_dict.update(dict(zip(df[smiles_col], df[name_col])))

    
    product_name_cols = [col for col in df.columns if col.startswith('Product_Name_')]
    product_smiles_cols = [col for col in df.columns if col.startswith('Product_SMILES_')]

    for name_col, smiles_col in zip(product_name_cols, product_smiles_cols):
        product_name_dict.update(dict(zip(df[smiles_col], df[name_col])))

    
    mol_system_dict = {**reactant_name_dict, **product_name_dict}

    return mol_system_dict

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:
def create_Gaussian_inputfile(name, smiles):
    
    if not isinstance(smiles, str) or pd.isna(smiles) or smiles.strip() == "":
        print("Public message removed for release.")
        return
        
    
    mol = Chem.MolFromSmiles(smiles)

    
    smiles_noH = MolToSmiles(mol, allHsExplicit=False)

    
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("smi", "gjf")
    mol = pybel.readstring("smi", smiles_noH)
    mol.addh()
    mol.make3D(forcefield='mmff94', steps=50)

    
    filename = name.replace(' ', '_') + '.gjf'

    
    obConversion.WriteFile(mol.OBMol, filename)

    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')

    with open(filename, 'r') as file:
        lines = file.readlines()

    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break

    
    if start_index is not None:
        with open(filename, 'w') as file:
            file.writelines(lines[start_index:])

    
    
    with open(filename, 'r') as f:
        coordinates = f.read()    

    
    
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, filename.replace('.gjf', '.chk'))

    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles)

    
    
    with open(filename, 'w') as f:
        f.write(f"%nprocshared={nproc}\n")
        f.write(f"%mem={mem}GB\n")
        f.write(f"%chk={chk_file}\n")
        f.write("# opt freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682\n\n")
        f.write(f"{filename} opt+freq gas\n\n")
        f.write(f"{charge} {multiplicity}\n")
        f.write(coordinates)
        
        
        f.write("\n\n")


In [ ]:

def create_or_load_molecule_xlsx(directory):
    
    xlsx_path = os.path.join(directory, 'molecule.xlsx')
    
    
    if not os.path.exists(xlsx_path):
        
        df = pd.DataFrame(columns=['FileName', 'SMILES'])
        
        df.to_excel(xlsx_path, index=False)
    else:
        
        df = pd.read_excel(xlsx_path)
    
    return df

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def normalization(mol):
    smi=Chem.MolToSmiles(mol)
    n_mol=Chem.MolFromSmiles(smi)
    return n_mol

In [ ]:
def normalization_SMILES(excel_path, SMILESName="SMILES"):
    
    df = pd.read_excel(excel_path)
    file_name_without_extension = get_filename_without_extension(excel_path)
    
    
    if SMILESName in df.columns:
        
        for index, row in df.iterrows():
            
            if pd.isna(row[SMILESName]):
                print("Public message removed for release.")
                continue  

            
            mol = Chem.MolFromSmiles(row[SMILESName])
            if mol is not None:  
                
                n_mol = normalization(mol)
                
                n_smi = Chem.MolToSmiles(n_mol)
                
                df.at[index, SMILESName] = n_smi
            else:
                print("Public message removed for release.")

        
        df.to_excel(f'{file_name_without_extension}.xlsx', index=False)
    else:
        print(f"The {SMILESName} column does not exist in the provided Excel file.")

In [ ]:
def normalize_all_smiles(df, mol_excel_path):
    
    
    smiles_columns = [col for col in df.columns if col.startswith('Reactant_SMILES_') or col.startswith('Product_SMILES_')]

    
    for smiles_col in smiles_columns:
        
        if df[smiles_col].dropna().empty:
            
            print("Public message removed for release.")
            continue
        else:
            
            normalization_SMILES(mol_excel_path, SMILESName=smiles_col)

In [ ]:

def read_excel_to_dict(excel_path, key_col, value_col):
    
    try:
        
        df = pd.read_excel(excel_path)
        
        
        if key_col not in df.columns or value_col not in df.columns:
            raise ValueError("Public message removed for release.")
        
        
        result_dict = pd.Series(df[value_col].values,index=df[key_col]).to_dict()
        
        return result_dict
    except FileNotFoundError:
        print("Public message removed for release.")
    except ValueError as e:
        print(e)
    except Exception as e:
        print("Public message removed for release.")

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:

def check_and_create_gaussian_database(gaussian_database_path=os.path.join(os.path.abspath(os.path.expanduser(result['gaussian_database_path'])), 'Gaussian_database')):
    
    home_directory = os.path.abspath(os.path.expanduser(result['gaussian_database_path']))
    
    optfreq_gaussian_database_path = os.path.join(home_directory, 'Gaussian_database/opt+freq')
    RESPpolymer_database_path = os.path.join(home_directory, 'Gaussian_database/RESPpolymer')
    
    
    if not os.path.exists(gaussian_database_path):
        
        os.makedirs(gaussian_database_path)
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(optfreq_gaussian_database_path) and os.path.exists(RESPpolymer_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    else:
        
        print("Public message removed for release.")
    return gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path

In [ ]:


def compare_smiles_dicts(system_dict, database_dict):
    
    not_found_molecule_dict = {}
    found_molecule_dict = {}

    
    for smiles, name in system_dict.items():
        if smiles not in database_dict.keys():
            not_found_molecule_dict[smiles] = name

    
    for smiles in database_dict:
        if smiles in system_dict:
            name = system_dict[smiles]  
            found_molecule_dict[smiles] = name

    return not_found_molecule_dict, found_molecule_dict

In [ ]:

def read_system_excel_to_dict(excel_path, SMILESName="SMILES", Name="Name"):
    
    return read_excel_to_dict(excel_path, SMILESName, Name)


def read_database_excel_to_dict(database_path):
    
    return read_excel_to_dict(database_path, 'SMILES', 'FileName')

In [ ]:
def copy_files_based_on_smiles(database_directory, database_excel_path, found_molecule_dict, destination_directory):
    
    
    if not os.path.exists(destination_directory):
        raise Exception(f"{destination_directory} does not exist")

    
    df = pd.read_excel(database_excel_path)

    
    for smiles, name in found_molecule_dict.items():
        
        matched_files = df[df['SMILES'] == smiles]['FileName'].tolist()
        
        
        for matched_file in matched_files:
            
            for ext in ['.gjf', '.chk', '.out']:
                file_to_copy = matched_file + ext
                source_path = os.path.join(database_directory, file_to_copy)
                
                if os.path.isfile(source_path):
                    
                    new_name = name + ext
                    destination_path = os.path.join(destination_directory, new_name)
                    
                    shutil.copy2(source_path, destination_path)
                    print("Public message removed for release.")
                else:
                    print("Public message removed for release.")

In [ ]:
from qc_database_utils import (
    add_and_normalize_smiles,
    compare_smiles_dicts,
    copy_files_based_on_smiles,
    ensure_database_excel,
    get_orca_database_paths,
    read_database_excel_to_dict,
)

if 'result' not in globals():
    from cemp_software_settings import load_and_apply_settings
    result = load_and_apply_settings()


In [ ]:



orca_database_path, optfreq_orca_database_path, RESPpolymer_database_path = get_orca_database_paths(result)


molecule_database_path = optfreq_orca_database_path
polymer_database_path = RESPpolymer_database_path


molecule_database_excel_path = ensure_database_excel(molecule_database_path)
polymer_database_excel_path = ensure_database_excel(polymer_database_path)


molecule_database = read_database_excel_to_dict(molecule_database_excel_path)  
polymer_database = read_database_excel_to_dict(polymer_database_excel_path)  


In [ ]:

normalize_all_smiles(df, mol_excel_path)

mol_system_dict = extract_mol_system_dict(df)


mol_not_found_dict, mol_found_dict = compare_smiles_dicts(mol_system_dict, molecule_database)

In [ ]:

os.makedirs('opt+freq/success', exist_ok=True)
os.makedirs('opt+freq/failure', exist_ok=True)

os.makedirs("opt+freq/imaginary_frequency", exist_ok=True)

base_dir = 'opt+freq'
success_dir = os.path.join(base_dir, 'success') 
failure_dir = os.path.join(base_dir, 'failure') 
imaginary_frequency_dir = os.path.join(base_dir, "imaginary_frequency") 

In [ ]:

copy_files_based_on_smiles(molecule_database_path, molecule_database_excel_path, mol_found_dict, success_dir)

In [ ]:


def create_input_files_for_missing_molecules(not_found_molecule_dict):
    
    
    for smiles, name in not_found_molecule_dict.items():
        
        
        create_Gaussian_inputfile(name, smiles)

In [ ]:
create_input_files_for_missing_molecules(mol_not_found_dict)